# Bin packing

Given `n items` with weights `w1,w2..wn` and an `arbitrary number of bins` with maximum carry weight C.

Determine the `lowest number of bins` that can `contain all the items` without exceeding their carry weight.

In [ ]:
from pulp import *

# data
bin_sets = [
    ("Set 1", 100, [70, 60, 50, 33, 33, 33, 11, 7, 3]),
    ("Set 2", 100, [99, 94, 79, 64, 50, 46, 43, 37, 32, 19, 18, 7, 6, 3]),
    ("Set 3", 100, [49, 41, 34, 33, 29, 26, 26, 22, 20, 19]),
    ("Set 4", 524, [442, 252, 252, 252, 252, 252, 252, 252, 127, 127, 127, 127, 127, 106, 106, 106, 106, 85, 84, 46, 37, 37, 12, 12, 12, 10, 10, 10, 10, 10, 10, 9, 9]),
]

# Iterate all sets
for set_name, maximum_bin_load, weights in bin_sets:
    n = len(weights) # number of total items

    # minimizing model
    model = LpProblem(sense=LpMinimize)

    # variables - binary x[i][j] based on if the i-th item is in the j-th bin
    variables = [
        [LpVariable(name=f"x_{i}_{j}", cat=LpBinary) for j in range(n)]
        for i in range(n)
    ]

    # ... and also the number of bins
    bin_count = LpVariable(name="bin count", cat=LpInteger)

    # inequalities
    ## each bin doesn't contain more than its maximum load
    for j in range(n):
        model += lpSum([variables[i][j] * weights[i] for i in range(n)]) <= maximum_bin_load

    ## each item must be placed in exactly one bin
    for i in range(n):
        model += lpSum([variables[i][j] for j in range(n)]) == 1

    ## we also want the bin count to be the highest bin number
    for i in range(n):
        for j in range(n):
            model += bin_count >= (j + 1) * variables[i][j]

    # objective function - number of bins
    model += bin_count

    status = model.solve(PULP_CBC_CMD(msg=False))

    print(set_name + "\n" + "-" * len(set_name))
    print("bin count:", int(bin_count.value()), ", max_weight", maximum_bin_load)
    for i in range(int(bin_count.value())):
        print(f"bin:{i}")        
        for j in range(n):
            for k in range(n):
                if (variables[j][k].value() != 0) & (i== k):
                    print(f"\t item {j} (weight {weights[j]})")  
    print()

Set 1
-----
bin count: 4 , max_weight 100
bin:0
	 item 2 (weight 50)
	 item 3 (weight 33)
bin:1
	 item 4 (weight 33)
	 item 5 (weight 33)
bin:2
	 item 0 (weight 70)
	 item 6 (weight 11)
	 item 7 (weight 7)
	 item 8 (weight 3)
bin:3
	 item 1 (weight 60)

Set 2
-----
bin count: 7 , max_weight 100
bin:0
	 item 3 (weight 64)
	 item 8 (weight 32)
bin:1
	 item 2 (weight 79)
	 item 10 (weight 18)
bin:2
	 item 4 (weight 50)
	 item 5 (weight 46)
	 item 13 (weight 3)
bin:3
	 item 7 (weight 37)
	 item 9 (weight 19)
	 item 11 (weight 7)
bin:4
	 item 0 (weight 99)
bin:5
	 item 1 (weight 94)
	 item 12 (weight 6)
bin:6
	 item 6 (weight 43)

Set 3
-----
bin count: 3 , max_weight 100
bin:0
	 item 1 (weight 41)
	 item 3 (weight 33)
	 item 6 (weight 26)
bin:1
	 item 0 (weight 49)
	 item 4 (weight 29)
	 item 7 (weight 22)
bin:2
	 item 2 (weight 34)
	 item 5 (weight 26)
	 item 8 (weight 20)
	 item 9 (weight 19)

